In [ ]:
import datetime
import time
from pathlib import Path
from collections import Counter
import csv
from typing import List, Optional
from kiwipiepy import Kiwi

In [32]:
import datetime
import time
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from typing import List, Optional
from kiwipiepy import Kiwi

def analyze_word_frequency(
    target_words: Optional[List[str]] = None, 
    output_filename: str = "v99_빈도분석.xlsx",
    sheet_name: str = "빈도분석 결과",
    verbose: bool = True
) -> None:
    """
    1. 數 ~ 10. 望 폴더 내의 Markdown 파일들을 읽어 형태소 분석을 수행하고,
    전체 빈도 및 파일별(1. 공리 ~ 53. 인간 등) 분포를 계산하여 Excel로 저장합니다.
    """
    # ⏱️ 전체 시작 시간 측정
    total_start_time = time.time()
    
    ROOT = Path(r"/home/yhsong/gdrive/내 드라이브/my_vault/essay_v99")

    TARGET_TAGS = {"NNG", "NNP", "VV", "VA", "MAG"}
    TAG_MAP = {
        "NNG": "일반명사",
        "NNP": "고유명사",
        "VV": "동사",
        "VA": "형용사",
        "MAG": "일반부사"
    }
    EXCLUDE_PATTERNS = ["집필 원칙", "목차", "단원별", "전체", "수식", "wordcount"]

    filter_words = set(target_words) if target_words else None

    if verbose:
        current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
        print(f"[{current_time}] 🚀 파일별(1. 공리 ~ 53. 인간) 형태소 분포 분석을 시작합니다...")

    if not ROOT.exists():
        raise FileNotFoundError(f"⚠️ 원본 폴더를 찾을 수 없습니다: {ROOT}")

    # 1부터 10까지의 챕터 폴더 매칭
    target_folders = []
    for num in range(1, 11):
        matched_dirs = list(ROOT.glob(f"{num}.*"))
        for d in matched_dirs:
            if d.is_dir():
                target_folders.append(d)

    # 지정된 폴더들 내부에서 .md 파일 수집
    file_list = []
    for folder in target_folders:
        for md in folder.rglob("*.md"):
            if not any(pat in md.name for pat in EXCLUDE_PATTERNS):
                file_list.append(md)
    
    # 💡 파일 이름 순서대로 정렬 (예: '1. 공리.md' -> '53. 인간.md' 순서 보장)
    # 파일명 앞의 숫자를 기준으로 자연스럽게 정렬되도록 순서 정렬을 가합니다.
    try:
        file_list.sort(key=lambda x: int(x.stem.split('.')[0]))
    except (ValueError, IndexError):
        file_list.sort(key=lambda x: x.name) # 예외 발생 시 이름순 정렬

    # 💡 컬럼 헤더로 사용할 파일명(확장자 제외) 리스트 생성
    file_names = [md.stem for md in file_list]
    total_files = len(file_list)

    # 구조화된 빈도 저장을 위한 딕셔너리
    # (어휘, 품사) -> { 파일명(stem): 빈도 }
    file_distribution = defaultdict(lambda: defaultdict(int))
    total_counter = Counter()  # (어휘, 품사) -> 전체 빈도

    kiwi = Kiwi()

    # 1. 파일별 분석 단계
    for idx, md in enumerate(file_list, 1):
        file_start_time = time.time()
        file_stem = md.stem  # 확장자를 뺀 파일명 (예: "1. 공리")
        
        text = md.read_text(encoding="utf-8")
        
        for token in kiwi.tokenize(text):
            if token.tag in TARGET_TAGS:
                if filter_words and token.form not in filter_words:
                    continue
                    
                key = (token.form, token.tag)
                total_counter[key] += 1
                file_distribution[key][file_stem] += 1
        
        if verbose:
            file_end_time = time.time()
            current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
            relative_path = md.relative_to(ROOT).as_posix()
            print(f"[{current_time}] 🔄 진행 중: ({idx}/{total_files}) {relative_path} 완료 | 소요 시간: {file_end_time - file_start_time:.4f}초")

    # 2. 데이터 정리 및 파일별 컬럼 배치 단계
    save_start_time = time.time()
    
    data = []
    # 전체 빈도가 높은 순서대로 정렬하여 추출
    for (form, tag), total_cnt in total_counter.most_common():
        korean_tag = TAG_MAP.get(tag, tag)
        
        # 기본 정보 열 구성
        row = {
            "어휘": form,
            "품사": korean_tag,
            "전체 빈도": total_cnt
        }
        
        # 각 파일별 빈도를 순서대로 열에 추가 (없으면 0)
        for f_name in file_names:
            row[f_name] = file_distribution[(form, tag)][f_name]
            
        data.append(row)
    
    # 데이터프레임 생성
    df = pd.DataFrame(data)
    
    if df.empty:
        columns = ["어휘", "품사", "전체 빈도"] + file_names
        df = pd.DataFrame(columns=columns)

    # Excel 저장
    df.to_excel(output_filename, index=False, sheet_name=sheet_name, engine="openpyxl")
            
    if verbose:
        save_end_time = time.time()
        current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
        print(f"[{current_time}] 💾 파일별 분포 포함 Excel 저장 완료 ({sheet_name} 시트) | 소요 시간: {save_end_time - save_start_time:.4f}초")

    # 3. 최종 요약 출력
    total_end_time = time.time()
    current_time = datetime.datetime.now().strftime('%Y%m%d %H:%M:%S')
    
    print("-" * 60)
    print(f"[{current_time}] 🎉 최종 완료: {output_filename}")
    print(f"📊 고유 형태소 수: {len(total_counter)}")
    print(f"📊 총 토큰 수: {sum(total_counter.values())}")
    print(f"⏱️ 총 실행 시간: {total_end_time - total_start_time:.2f}초")
    print("-" * 60)


# =====================================================================
# 실행
# =====================================================================
if __name__ == "__main__":
    analyze_word_frequency()

In [33]:
words = ["세계", "우주", "자연", "현실", "실재"]
words = None

output_filename"/home/yhsong/gdrive/내 드라이브/my_vault/v99_핵심단어_분석.xlsx"
analyze_word_frequency(words, 
    output_filename="",
    sheet_name="부사_및_명사_추출",
    verbose=False)

------------------------------------------------------------
[20260524 23:29:17] 🎉 최종 완료: /home/yhsong/gdrive/내 드라이브/my_vault/v99_핵심단어_분석.xlsx
📊 고유 형태소 수: 3364
📊 총 토큰 수: 19280
⏱️ 총 실행 시간: 7.40초
------------------------------------------------------------
